In [ ]:
# CELL 7 — Feature extraction: L1 file handwriting, L2 compression fingerprints, L3 consistency, L4 revision diff
!pip install -q pikepdf pebble

import os, re, glob, zlib, math, logging, collections
import numpy as np
import pandas as pd
import pikepdf
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pikepdf").setLevel(logging.ERROR)

cands = ["/kaggle/working/pilot/manifest.csv"] + sorted(glob.glob("/kaggle/input/**/pilot/manifest.csv", recursive=True))
MANIFEST = next((c for c in cands if os.path.exists(c)), None)
assert MANIFEST, "manifest.csv not found - tell Claude where the pilot folder is"
FILES = os.path.join(os.path.dirname(MANIFEST), "files")
OUT_CSV = "/kaggle/working/pilot_features.csv"
man = pd.read_csv(MANIFEST)
missing = [f for f in man.out_file if not os.path.exists(os.path.join(FILES, f))]
print("Manifest:", MANIFEST, "| rows:", len(man), "| missing files:", len(missing))
assert not missing, f"Missing files, e.g. {missing[:3]}"

MAX_STREAMS = 120
TIME_LIMIT = 60
TOOL_WORDS = [b"pypdf", b"pymupdf", b"mupdf", b"pikepdf", b"qpdf"]
STREAM_BODY = re.compile(rb"(?<!end)stream\r?\n.*?endstream", re.S)

def ratio(a, b):
    return a / b if b else 0.0

def file_features(raw):
    f = {}
    skel = STREAM_BODY.sub(b"stream endstream", raw)
    f["size_log"] = math.log10(len(raw) + 1)
    m = re.match(rb"\s*%PDF-(\d)\.(\d)", raw[:64])
    f["version"] = int(m.group(1)) + int(m.group(2)) / 10 if m else 0
    f["binary_comment"] = int(bool(re.match(rb"\s*%PDF-\d\.\d\s*[\r\n]+%[\x80-\xff]{4}", raw[:40])))
    f["n_eof"] = raw.count(b"%%EOF")
    f["n_startxref"] = raw.count(b"startxref")
    f["xref_table"] = int(bool(re.search(rb"(^|[\r\n])xref\s*[\r\n]", skel)))
    f["xref_stream"] = int(bool(re.search(rb"/Type\s*/XRef", skel)))
    f["obj_streams"] = int(bool(re.search(rb"/Type\s*/ObjStm", skel)))
    f["linearized"] = int(b"/Linearized" in raw[:2048])
    f["true_revisions"] = f["n_eof"] - f["linearized"]
    crlf = skel.count(b"\r\n")
    cr = skel.count(b"\r") - crlf
    lf = skel.count(b"\n") - crlf
    tot = crlf + cr + lf
    f["eol_crlf"], f["eol_cr"], f["eol_lf"] = ratio(crlf, tot), ratio(cr, tot), ratio(lf, tot)
    objs = re.findall(rb"(\d+)\s+(\d+)\s+obj", skel)
    nums = [int(a) for a, _ in objs]
    f["n_obj_defs"] = len(nums)
    f["obj_order_increasing"] = ratio(sum(1 for a, b in zip(nums, nums[1:]) if b > a), max(len(nums) - 1, 1))
    f["obj_gen_nonzero"] = ratio(sum(1 for _, g in objs if g != b"0"), len(objs))
    f["obj_max_over_count"] = ratio(max(nums) if nums else 0, len(set(nums)))
    f["obj_redefined"] = len(nums) - len(set(nums))
    f["obj_newline_dict"] = ratio(len(re.findall(rb"obj\s*[\r\n]+\s*<<", skel)), len(nums))
    f["dict_open_space"] = ratio(skel.count(b"<< /"), skel.count(b"<<"))
    f["dict_close_space"] = ratio(skel.count(b" >>"), skel.count(b">>"))
    nstream = len(re.findall(rb"(?<!end)stream", skel))
    f["stream_kw_crlf"] = ratio(len(re.findall(rb"(?<!end)stream\r\n", raw)), nstream)
    f["endobj_newline_before"] = ratio(len(re.findall(rb"[\r\n]endobj", skel)), skel.count(b"endobj"))
    f["reals_dot_zero"] = ratio(len(re.findall(rb"\s\d+\.0[\s/\]>]", skel)), len(re.findall(rb"\s\d+\.\d+[\s/\]>]", skel)))
    f["has_ID"] = int(b"/ID" in skel)
    f["tool_word_in_file"] = int(any(w in skel.lower() for w in TOOL_WORDS))
    return f

def stream_sig(s):
    d = {}
    flt = s.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    d["filter"] = "+".join(names) if names else "none"
    d["has_parms"] = int(s.get("/DecodeParms") is not None)
    d["len_indirect"] = int(bool(getattr(s.get("/Length"), "is_indirect", False)))
    d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["ratio"] = -1, 0, 0, 0, 0, 0.0
    raw = s.read_raw_bytes()
    if names == ["/FlateDecode"] and len(raw) >= 6:
        d["hdr"] = raw[0] * 256 + raw[1]
        try:
            dob = zlib.decompressobj()
            dec = dob.decompress(raw)
            body = raw[:len(raw) - len(dob.unused_data)]
            d["tail_junk"] = int(len(dob.unused_data) > 0)
            d["ratio"] = ratio(len(body), len(dec))
            d["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
            d["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
            if len(dec) <= 2_000_000:
                for lv in range(1, 10):
                    if zlib.compress(dec, lv) == body:
                        d["zlevel"] = lv
                        break
        except Exception:
            d["hdr"] = -2
    return d

def extract(name):
    path = os.path.join(FILES, name)
    with open(path, "rb") as fh:
        raw = fh.read()
    f = {"out_file": name}
    f.update(file_features(raw))
    with pikepdf.open(path) as pdf:
        info = pdf.docinfo
        prod = str(info.get("/Producer", "")) if info is not None else ""
        f["has_producer"] = int(bool(prod.strip()))
        f["producer_len"] = len(prod)
        cd, md = str(info.get("/CreationDate", "")), str(info.get("/ModDate", ""))
        f["moddate_eq_creation"] = int(bool(cd) and cd == md)
        f["has_xmp"] = int("/Metadata" in pdf.Root)
        f["n_pages"] = len(pdf.pages)

        streams = []
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream):
                    streams.append((pi, s))
            if len(streams) >= MAX_STREAMS:
                break
        streams = streams[:MAX_STREAMS]
        sigs = []
        for pi, s in streams:
            try:
                sigs.append((pi, getattr(s, "objgen", (-1, 0))[0], stream_sig(s)))
            except Exception:
                pass
        cs_nums = {num for _, num, _ in sigs}

    n = len(sigs)
    f["cs_n"] = n
    if n:
        fl = [d["filter"] for _, _, d in sigs]
        f["cs_frac_flate"] = ratio(fl.count("/FlateDecode"), n)
        f["cs_frac_lzw"] = ratio(sum("LZW" in x for x in fl), n)
        f["cs_frac_none"] = ratio(fl.count("none"), n)
        f["cs_n_filters"] = len(set(fl))
        flate = [d for _, _, d in sigs if d["filter"] == "/FlateDecode"]
        f["cs_frac_zlib_repro"] = ratio(sum(d["zlevel"] > 0 for d in flate), len(flate)) if flate else np.nan
        f["cs_n_hdr"] = len({d["hdr"] for _, _, d in sigs})
        f["cs_n_zlevel"] = len({d["zlevel"] for _, _, d in sigs})
        f["cs_frac_sync"] = np.mean([d["sync_end"] for _, _, d in sigs])
        f["cs_frac_adler"] = np.mean([d["adler_ok"] for _, _, d in sigs])
        f["cs_frac_junk"] = np.mean([d["tail_junk"] for _, _, d in sigs])
        f["cs_frac_len_indirect"] = np.mean([d["len_indirect"] for _, _, d in sigs])
        r = np.array([d["ratio"] for _, _, d in sigs])
        f["cs_ratio_std"] = float(r.std())
        f["cs_ratio_max_z"] = float(np.max(np.abs(r - r.mean())) / r.std()) if r.std() > 0 else 0.0
        keys = [(d["filter"], d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["has_parms"], d["len_indirect"]) for _, _, d in sigs]
        cnt = collections.Counter(keys)
        f["cs_n_sig"] = len(cnt)
        f["cs_majority_share"] = max(cnt.values()) / n
        f["cs_n_singletons"] = sum(1 for v in cnt.values() if v == 1) if n >= 3 else 0
        odd = [pi for (pi, _, _), k in zip(sigs, keys) if cnt[k] == 1] if n >= 3 else []
        f["loc_page"] = odd[0] if len(odd) == 1 else -1

    f.update({"upd_n_objs": 0, "upd_touch_content": 0, "upd_has_annot": 0, "upd_has_page": 0, "upd_has_sig": 0})
    if f["true_revisions"] > 1:
        ends = [m.end() for m in re.finditer(rb"%%EOF", raw)]
        tail = STREAM_BODY.sub(b"stream endstream", raw[ends[-2]:])
        objs = re.findall(rb"(\d+)\s+\d+\s+obj(.*?)endobj", tail, re.S)
        f["upd_n_objs"] = len(objs)
        for num, body in objs:
            if int(num) in cs_nums:
                f["upd_touch_content"] = 1
            if re.search(rb"/Type\s*/Annot|/Subtype\s*/Text\b", body):
                f["upd_has_annot"] = 1
            if re.search(rb"/Type\s*/Page\b", body):
                f["upd_has_page"] = 1
            if b"/ByteRange" in body:
                f["upd_has_sig"] = 1
    return f

rows, errors = [], []
with ProcessPool(max_workers=4) as pool:
    futs = {pool.schedule(extract, args=(n,), timeout=TIME_LIMIT): n for n in man.out_file}
    for i, fu in enumerate(tqdm(as_completed(futs), total=len(futs), desc="Files")):
        name = futs[fu]
        try:
            rows.append(fu.result())
        except TimeoutError:
            errors.append((name, "Timeout"))
        except Exception as ex:
            errors.append((name, type(ex).__name__))
        if (i + 1) % 300 == 0:
            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

feat = man.merge(pd.DataFrame(rows), on="out_file", how="left")
feat.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV, "| rows:", len(feat), "| feature columns:", feat.shape[1] - man.shape[1])
print("Files that failed:", len(errors), pd.Series([e for _, e in errors]).value_counts().to_dict() if errors else "")

g0 = feat[feat.variant == "G0_original"]
print("\n=== CHECK 1: old LZW compression in original page content ===")
print("Originals with any LZW content stream:", round((g0.cs_frac_lzw > 0).mean() * 100, 1), "%")
print("Originals fully Flate:", round((g0.cs_frac_flate == 1).mean() * 100, 1), "%")
print("\n=== CHECK 2: can Python zlib exactly reproduce the original Adobe compression? ===")
print("Avg share of original Flate content streams reproducible by zlib:", round(g0.cs_frac_zlib_repro.mean() * 100, 1), "%")
print("By family:")
print(g0.groupby("family")["cs_frac_zlib_repro"].mean().mul(100).round(1))

cols = ["true_revisions", "cs_n_sig", "cs_n_singletons", "cs_majority_share", "cs_frac_zlib_repro",
        "cs_frac_lzw", "upd_touch_content", "upd_has_annot", "tool_word_in_file"]
print("\n=== PREVIEW: average of key features per version ===")
print(feat.groupby("variant")[cols].mean().round(3).to_string())

ed = feat[feat.variant.isin(["F1_edit_resave", "F2_edit_recompress"]) & feat.edit_page.notna()]
hit = ed[ed.loc_page >= 0]
print("\n=== PREVIEW: localization (does the odd-one-out stream point to the edited page?) ===")
print("Forgeries where one stream stood out:", len(hit), "of", len(ed))
print("Of those, correct page:", round((hit.loc_page == hit.edit_page).mean() * 100, 1) if len(hit) else 0, "%")

In [ ]:
# CELL 8 — Pilot experiments: baselines vs our model, per-pair / per-tool / per-family results, leave-one-tool-out
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")

df = pd.read_csv("/kaggle/working/pilot_features.csv")
print("Rows:", len(df), "| train:", int((df.split == "train").sum()), "| test:", int((df.split == "test").sum()))

GROUPS = {
    "META (metadata only)": ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"],
    "L1 (file style)": ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream",
                        "obj_streams", "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs",
                        "obj_order_increasing", "obj_gen_nonzero", "obj_max_over_count", "obj_redefined",
                        "obj_newline_dict", "dict_open_space", "dict_close_space", "stream_kw_crlf",
                        "endobj_newline_before", "reals_dot_zero", "has_ID", "tool_word_in_file", "n_pages"],
    "L2+L3 (compression + consistency)": ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters",
                        "cs_frac_zlib_repro", "cs_n_hdr", "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk",
                        "cs_frac_len_indirect", "cs_ratio_std", "cs_ratio_max_z", "cs_n_sig", "cs_majority_share",
                        "cs_n_singletons"],
    "L4 (last-update diff)": ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"],
}
ALL = sorted(set(sum(GROUPS.values(), [])))
GROUPS["FULL MODEL (all layers)"] = ALL
missing = [c for c in ALL if c not in df.columns]
assert not missing, f"Missing feature columns: {missing}"

PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

def tpr_at(y, s, fpr):
    thr = np.quantile(s[y == 0], 1 - fpr)
    return float((s[y == 1] > thr).mean())

def scores(y, s):
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    try:
        auc = roc_auc_score(y, s)
    except ValueError:
        auc = np.nan
    return {"ROC-AUC": round(auc, 3), "PR-AUC": round(average_precision_score(y, s), 3),
            "Detect@1%FalseAlarm": round(tpr_at(y, s, 0.01), 3), "Detect@5%FalseAlarm": round(tpr_at(y, s, 0.05), 3)}

def auc_or_nan(y, s):
    try:
        return round(roc_auc_score(y, s), 3)
    except ValueError:
        return np.nan

tr = df[df.split == "train"].reset_index(drop=True)
te = df[df.split == "test"].reset_index(drop=True)

def fit(cols, train):
    return lgb.LGBMClassifier(**PARAMS).fit(train[cols], train["label"])

# ---------- E1: main results table ----------
rows = []
rows.append({"Method": "Rule: %%EOF count > 1 (industry check)", **scores(te.label, te.n_eof)})
rows.append({"Method": "Rule: true revisions > 1 (linearization-corrected)", **scores(te.label, te.true_revisions)})
models = {}
for gname, cols in GROUPS.items():
    m = fit(cols, tr)
    models[gname] = m
    rows.append({"Method": f"LightGBM: {gname}", **scores(te.label, m.predict_proba(te[cols])[:, 1])})
print("\n=== E1: MAIN RESULTS (test set, unseen documents) ===")
print(pd.DataFrame(rows).to_string(index=False))

# ---------- threshold chosen on TRAIN only (5% false alarms, out-of-fold, grouped by document) ----------
oof = np.zeros(len(tr))
for a, b in GroupKFold(n_splits=5).split(tr, tr.label, tr.orig_file):
    oof[b] = fit(ALL, tr.iloc[a]).predict_proba(tr.iloc[b][ALL])[:, 1]
THR = float(np.quantile(oof[tr.label.values == 0], 0.95))
te["score"] = models["FULL MODEL (all layers)"].predict_proba(te[ALL])[:, 1]
te["flagged"] = (te.score > THR).astype(int)
print(f"\nThreshold (set on training data for ~5% false alarms): {THR:.3f}")

# ---------- E2: per version ----------
print("\n=== E2: FLAG RATE PER VERSION (genuine rows = false alarms, fake rows = detections) ===")
pv = te.groupby(["variant", "label"])["flagged"].mean().mul(100).round(1).reset_index()
pv.columns = ["variant", "label", "flagged_%"]
print(pv.to_string(index=False))

print("\n=== E2b: SEPARATION PER MATCHED PAIR (ROC-AUC) ===")
for g, f in [("G1_resave", "F1_edit_resave"), ("G2_recompress", "F2_edit_recompress"), ("G3_annot_incr", "F3_edit_incr"),
             ("G0_original", "F1_edit_resave")]:
    sub = te[te.variant.isin([g, f])]
    print(f"{g:15s} vs {f:20s}: AUC = {auc_or_nan(sub.label, sub.score)}")

# ---------- E3: per tool (full-rewrite pairs only) ----------
print("\n=== E3: PER TOOL, full-rewrite pairs (ROC-AUC) ===")
for t in ["pikepdf", "pymupdf", "pypdf"]:
    a1 = te[(te.tool == t) & te.variant.isin(["G1_resave", "F1_edit_resave"])]
    a2 = te[(te.tool == t) & te.variant.isin(["G2_recompress", "F2_edit_recompress"])]
    print(f"{t:8s} | resave (G1 vs F1): {auc_or_nan(a1.label, a1.score)} (n={len(a1)})"
          f" | recompress (G2 vs F2): {auc_or_nan(a2.label, a2.score)} (n={len(a2)})")

# ---------- E4: per family ----------
print("\n=== E4: PER ISSUER FAMILY (ROC-AUC) ===")
for fam in sorted(te.family.unique()):
    s_all = te[te.family == fam]
    s_rw = s_all[s_all.variant.isin(["G1_resave", "F1_edit_resave", "G2_recompress", "F2_edit_recompress"])]
    print(f"{fam:28s} | all versions: {auc_or_nan(s_all.label, s_all.score)} | full rewrites only: {auc_or_nan(s_rw.label, s_rw.score)}")

# ---------- E5: leave-one-tool-out (generalization to an unseen editing tool) ----------
print("\n=== E5: LEAVE-ONE-TOOL-OUT (train without the tool, test only on it; full-rewrite pairs) ===")
RW = ["G0_original", "G1_resave", "F1_edit_resave", "G2_recompress", "F2_edit_recompress"]
for t in ["pikepdf", "pymupdf", "pypdf"]:
    tr_t = tr[tr.variant.isin(RW) & (tr.tool != t)]
    te_t = te[te.variant.isin(["G1_resave", "F1_edit_resave", "G2_recompress", "F2_edit_recompress"]) & (te.tool == t)]
    m = fit(ALL, tr_t)
    print(f"unseen tool = {t:8s}: AUC = {auc_or_nan(te_t.label, m.predict_proba(te_t[ALL])[:, 1])} (test n={len(te_t)})")

# ---------- top features ----------
imp = pd.Series(models["FULL MODEL (all layers)"].booster_.feature_importance("gain"), index=ALL)
print("\n=== TOP 15 FEATURES (full model, by gain) ===")
print((imp / imp.sum() * 100).sort_values(ascending=False).head(15).round(1).to_string())

te.to_csv("/kaggle/working/pilot_test_predictions.csv", index=False)
print("\nSaved: /kaggle/working/pilot_test_predictions.csv")

In [ ]:
# CELL 9 — Dataset v2 generator: 15 versions per document (tool-native edits, real laundering, harder honest twins)
!pip install -q pikepdf pymupdf pypdf pebble

import os, io, re, glob, json, zlib, shutil, random, logging
import pandas as pd
import pikepdf
import pymupdf as fitz
from pypdf import PdfWriter
from pypdf.generic import ArrayObject, StreamObject, NameObject
from PIL import Image, ImageDraw, ImageFont
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pypdf").setLevel(logging.ERROR)
logging.getLogger("pikepdf").setLevel(logging.ERROR)

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

CLEAN_POOL = find_one("/kaggle/input/**/clean_pool.csv")
V1_MANIFEST = find_one("/kaggle/input/**/pilot/manifest.csv")
assert CLEAN_POOL and V1_MANIFEST, "Input files not found - is PDP BPTP attached as input?"
SRC = os.path.join(os.path.dirname(CLEAN_POOL), "govdocs_pdfs")
assert os.path.isdir(SRC), f"govdocs_pdfs not found next to {CLEAN_POOL}"
OUT = "/kaggle/working/pilot_v2/files"
META = "/kaggle/working/pilot_v2/meta"
os.makedirs(OUT, exist_ok=True)
os.makedirs(META, exist_ok=True)
print("Originals:", SRC)
print("v1 manifest:", V1_MANIFEST)

PER_FAMILY = 80
TIME_LIMIT = 180
FAMILIES = ["acrobat distiller windows", "acrobat distiller for", "acrobat pdfwriter for",
            "adobe pdf library", "corel pdf engine"]
TOOLS = ["pikepdf", "pymupdf", "pypdf"]

LIT = re.compile(rb"\((?:[^()\\]|\\.)*\)", re.S)
TAIL = re.compile(rb"(?:\s+|-?(?:\d+\.?\d*|\.\d+)(?=[\s(\]])|\((?:[^()\\]|\\.)*\))*\s*(?:\]\s*TJ|Tj|'|\")")
VALUE = re.compile(rb"\d[\d,\.]*\d")

def make_stamp(text):
    img = Image.new("RGBA", (420, 140), (255, 255, 255, 0))
    dr = ImageDraw.Draw(img)
    dr.rectangle([4, 4, 415, 135], outline=(200, 0, 0, 255), width=8)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 60)
    except Exception:
        font = ImageFont.load_default()
    dr.text((24, 34), text, fill=(200, 0, 0, 255), font=font)
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()

PNG_HONEST = make_stamp("RECEIVED")
PNG_FAKE = make_stamp("PAID")

# ---------- finding numbers in page code ----------
def digit_positions(seg):
    pos, i, n = [], 1, len(seg) - 1
    while i < n:
        b = seg[i]
        if b == 0x5C:
            j = i + 1
            if j < n and 48 <= seg[j] <= 55:
                k = j
                while k < n and k < j + 3 and 48 <= seg[k] <= 55:
                    k += 1
                i = k
            else:
                i = j + 1
            continue
        if 48 <= b <= 57:
            pos.append(i)
        i += 1
    return pos

def candidates(data):
    digits, values = [], []
    for m in LIT.finditer(data):
        if not TAIL.match(data[m.end():m.end() + 300]):
            continue
        seg = m.group()
        digits += [m.start() + p for p in digit_positions(seg)]
        for v in VALUE.finditer(seg):
            if v.start() > 0 and seg[v.start() - 1] == 0x5C:
                continue
            if sum(c in b"0123456789" for c in v.group()) >= 3:
                values.append((m.start() + v.start(), m.start() + v.end()))
    return digits, values

def new_value(old, rng):
    s = old.decode("latin-1")
    two_dec = s.count(".") == 1 and len(s.split(".")[1]) == 2
    intpart = s.split(".")[0] if two_dec else s
    nd = max(1, sum(c.isdigit() for c in intpart) + rng.choice([-1, 1, 2]))
    n = rng.randint(10 ** (nd - 1), 10 ** nd - 1)
    out = (f"{n:,}" if "," in s else str(n)) + (f".{rng.randint(0, 99):02d}" if two_dec else "")
    if out == s:
        out = str(n + 1)
    return out.encode()

def pk_streams(pdf, i):
    c = pdf.pages[i].obj.get("/Contents")
    if c is None:
        return []
    return [s for s in (list(c) if isinstance(c, pikepdf.Array) else [c]) if isinstance(s, pikepdf.Stream)]

def choose_specs(src, rng):
    dspec = vspec = None
    with pikepdf.open(src) as p:
        order = list(range(len(p.pages)))
        rng.shuffle(order)
        for i in order:
            for k, s in enumerate(pk_streams(p, i)):
                try:
                    data = s.read_bytes()
                except Exception:
                    continue
                d, v = candidates(data)
                if dspec is None and d:
                    pos = rng.choice(d)
                    old = data[pos:pos + 1]
                    new = bytes([rng.choice([c for c in b"0123456789" if c != old[0]])])
                    dspec = dict(page=i, k=k, start=pos, end=pos + 1, old=old, new=new)
                if vspec is None and v:
                    st, en = rng.choice(v)
                    vspec = dict(page=i, k=k, start=st, end=en, old=data[st:en], new=new_value(data[st:en], rng))
                if dspec and vspec:
                    return dspec, vspec
    return dspec, vspec

def apply_spec(data, spec):
    if data[spec["start"]:spec["end"]] != spec["old"]:
        raise RuntimeError("edit location mismatch")
    return data[:spec["start"]] + spec["new"] + data[spec["end"]:]

# ---------- tool-native edits ----------
def edit_pikepdf(pdf, spec):
    s = pk_streams(pdf, spec["page"])[spec["k"]]
    s.write(apply_spec(s.read_bytes(), spec))

def edit_fitz(doc, spec):
    x = doc[spec["page"]].get_contents()[spec["k"]]
    doc.update_stream(x, apply_spec(doc.xref_stream(x), spec))

def py_streams(page):
    c = page.get("/Contents")
    if c is None:
        return []
    c = c.get_object()
    out = []
    for it in (list(c) if isinstance(c, ArrayObject) else [c]):
        o = it.get_object()
        if isinstance(o, StreamObject):
            out.append(o)
    return out

def edit_pypdf(w, spec):
    o = py_streams(w.pages[spec["page"]])[spec["k"]]
    new = apply_spec(o.get_data(), spec)
    flt = o.get("/Filter")
    if flt is not None and flt != "/FlateDecode" and list(flt) != ["/FlateDecode"]:
        o[NameObject("/Filter")] = NameObject("/FlateDecode")
    if "/DecodeParms" in o:
        del o["/DecodeParms"]
    o.set_data(new)

def run_tool(tool, src, out, spec, mode):
    if tool == "pikepdf":
        with pikepdf.open(src) as p:
            if spec:
                edit_pikepdf(p, spec)
            if mode == "recompress":
                p.save(out, compress_streams=True, recompress_flate=True,
                       object_stream_mode=pikepdf.ObjectStreamMode.generate)
            elif mode == "linearize":
                p.save(out, linearize=True)
            else:
                p.save(out)
    elif tool == "pymupdf":
        d = fitz.open(src)
        if spec:
            edit_fitz(d, spec)
        if mode == "recompress":
            for pg in d:
                for x in pg.get_contents():
                    d.update_stream(x, d.xref_stream(x))
            d.save(out, garbage=4, deflate=True)
        else:
            d.save(out)
        d.close()
    else:
        w = PdfWriter(clone_from=src)
        if spec:
            edit_pypdf(w, spec)
        if mode == "recompress":
            for pg in w.pages:
                pg.compress_content_streams()
        with open(out, "wb") as f:
            w.write(f)

def fitz_incr(src, out, action):
    shutil.copy(src, out)
    d = fitz.open(out)
    try:
        if d.is_repaired:
            raise RuntimeError("file needed repair")
        res = action(d)
        d.saveIncr()
        return res
    finally:
        d.close()

def fitz_full(src, out, action):
    d = fitz.open(src)
    try:
        res = action(d)
        d.save(out)
        return res
    finally:
        d.close()

def add_note(d, i):
    r = d[i].rect
    d[i].add_text_annot(fitz.Point(r.x0 + 36, r.y0 + 36), "Reviewed")

def add_stamp(d, i, png):
    r = d[i].rect
    d[i].insert_image(fitz.Rect(r.x1 - 210, r.y0 + 40, r.x1 - 40, r.y0 + 97), stream=png, overlay=True)

def add_textbox(d, i, text):
    r = d[i].rect
    d[i].add_freetext_annot(fitz.Rect(r.x0 + 40, r.y1 - 80, r.x0 + 300, r.y1 - 58), text, fontsize=11)

def cover_replace(d, rng):
    order = list(range(len(d)))
    rng.shuffle(order)
    for i in order:
        words = [w for w in d[i].get_text("words") if re.search(r"\d", w[4]) and len(w[4]) >= 2]
        if not words:
            continue
        x0, y0, x1, y1, txt = rng.choice(words)[:5]
        r = fitz.Rect(x0, y0, x1, y1)
        if r.is_empty or r.height < 3:
            continue
        new = "".join(rng.choice("0123456789") if c.isdigit() else c for c in txt)
        if new == txt:
            new = txt[:-1] + ("7" if txt[-1] != "7" else "3")
        d[i].draw_rect(r, color=None, fill=(1, 1, 1), overlay=True)
        d[i].insert_text(fitz.Point(x0, y1 - r.height * 0.2), new, fontsize=max(5, r.height * 0.8),
                         fontname="helv", color=(0, 0, 0))
        return {"page": i, "old": txt, "new": new}
    raise RuntimeError("no numeric word for cover-and-replace")

def cleanup(stem):
    for p in glob.glob(os.path.join(OUT, f"{stem}__*.pdf")):
        os.remove(p)
    mp = os.path.join(META, stem + ".json")
    if os.path.exists(mp):
        os.remove(mp)

# ---------- one original -> 15 versions ----------
def process(job):
    name, fam = job
    fitz.TOOLS.mupdf_display_errors(False)
    stem = name[:-4]
    src = os.path.join(SRC, name)
    meta_path = os.path.join(META, stem + ".json")
    if os.path.exists(meta_path):
        with open(meta_path) as fh:
            rows = json.load(fh)
        if all(os.path.exists(os.path.join(OUT, r["out_file"])) for r in rows):
            return name, rows, None
    cleanup(stem)
    try:
        rng = random.Random(zlib.crc32(name.encode()) + 2026)
        split = "train" if random.Random(zlib.crc32(name.encode()) + 1).random() < 0.7 else "test"
        t_main, t_comp = rng.choice(TOOLS), rng.choice(TOOLS)
        dspec, vspec = choose_specs(src, rng)
        if dspec is None or vspec is None:
            raise RuntimeError("no editable number")
        P = lambda v: os.path.join(OUT, f"{stem}__{v}.pdf")
        pg = dspec["page"]

        shutil.copy(src, P("G0_original"))
        run_tool(t_main, src, P("G1_resave"), None, "resave")
        run_tool(t_comp, src, P("G2_recompress"), None, "recompress")
        fitz_incr(src, P("G3_note_incr"), lambda d: add_note(d, pg))
        run_tool("pikepdf", src, P("G4_linearize"), None, "linearize")
        fitz_full(src, P("G5_stamp_honest"), lambda d: add_stamp(d, pg, PNG_HONEST))
        fitz_incr(src, P("G6_textbox_honest"), lambda d: add_textbox(d, pg, "Approved - HR dept."))

        run_tool(t_main, src, P("F1_digit_resave"), dspec, "resave")
        run_tool(t_comp, src, P("F2_digit_recompress"), dspec, "recompress")
        fitz_incr(src, P("F3_digit_incr"), lambda d: edit_fitz(d, dspec))
        run_tool(t_main, src, P("F4_value_resave"), vspec, "resave")
        cov = fitz_full(src, P("F5_cover_replace"), lambda d: cover_replace(d, rng))
        run_tool("pikepdf", src, P("F6_digit_linearize"), dspec, "linearize")
        fitz_full(src, P("F7_stamp_fake"), lambda d: add_stamp(d, pg, PNG_FAKE))
        fitz_incr(src, P("F8_textbox_fake"), lambda d: add_textbox(d, pg, "TOTAL DUE: $98,450.00"))

        dold, dnew = dspec["old"].decode("latin-1"), dspec["new"].decode("latin-1")
        vold, vnew = vspec["old"].decode("latin-1"), vspec["new"].decode("latin-1")
        spec = [
            ("G0_original", 0, "none", "original", "none", None, None, None, "G0"),
            ("G1_resave", 0, t_main, "full", "none", None, None, None, "G1"),
            ("G2_recompress", 0, t_comp, "full_recompress", "none", None, None, None, "G2"),
            ("G3_note_incr", 0, "pymupdf", "incremental", "note", pg, None, None, "G3"),
            ("G4_linearize", 0, "pikepdf", "linearize", "none", None, None, None, "G4"),
            ("G5_stamp_honest", 0, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("G6_textbox_honest", 0, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
            ("F1_digit_resave", 1, t_main, "full", "digit", pg, dold, dnew, "G1"),
            ("F2_digit_recompress", 1, t_comp, "full_recompress", "digit", pg, dold, dnew, "G2"),
            ("F3_digit_incr", 1, "pymupdf", "incremental", "digit", pg, dold, dnew, "G3"),
            ("F4_value_resave", 1, t_main, "full", "value", vspec["page"], vold, vnew, "G1"),
            ("F5_cover_replace", 1, "pymupdf", "full", "cover", cov["page"], cov["old"], cov["new"], "G1"),
            ("F6_digit_linearize", 1, "pikepdf", "linearize", "digit", pg, dold, dnew, "G4"),
            ("F7_stamp_fake", 1, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("F8_textbox_fake", 1, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
        ]
        rows = [{"out_file": f"{stem}__{v}.pdf", "orig_file": name, "family": fam, "variant": v,
                 "label": lab, "tool": tool, "save_mode": mode, "edit_type": et, "edit_page": ep,
                 "old_value": ov, "new_value": nv, "twin": tw, "split": split}
                for v, lab, tool, mode, et, ep, ov, nv, tw in spec]
        with open(meta_path, "w") as fh:
            json.dump(rows, fh)
        return name, rows, None
    except Exception as ex:
        cleanup(stem)
        return name, None, f"{type(ex).__name__}: {str(ex)[:50]}"

# ---------- which originals: v1's 397 first, then spares ----------
v1 = pd.read_csv(V1_MANIFEST).drop_duplicates("orig_file")[["orig_file", "family"]]
pool = pd.read_csv(CLEAN_POOL)
jobs = []
for fam in FAMILIES:
    first = v1.loc[v1.family == fam, "orig_file"].tolist()
    taken = set(first)
    extra = [n for n in pool.loc[pool.producer_family == fam, "file"].sample(frac=1, random_state=7).tolist()
             if n not in taken]
    jobs += [(n, fam) for n in first + extra[:40]]
print("Originals to try:", len(jobs), "| v2 files already on disk:", len(glob.glob(os.path.join(OUT, "*.pdf"))))

results = {}
with ProcessPool(max_workers=4) as pool_ex:
    futs = {pool_ex.schedule(process, args=(job,), timeout=TIME_LIMIT): job for job in jobs}
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Originals"):
        job = futs[fu]
        try:
            _, rows, err = fu.result()
        except TimeoutError:
            rows, err = None, "Timeout"
            cleanup(job[0][:-4])
        except Exception as ex:
            rows, err = None, type(ex).__name__
            cleanup(job[0][:-4])
        results[job] = (rows, err)

kept = {f: 0 for f in FAMILIES}
final_rows, errors = [], []
for job in jobs:
    rows, err = results.get(job, (None, "missing"))
    if rows is None:
        errors.append(str(err).split(":")[0] + (": " + str(err).split(":", 1)[1].strip() if ":" in str(err) else ""))
        continue
    if kept[job[1]] >= PER_FAMILY:
        cleanup(job[0][:-4])
        continue
    kept[job[1]] += 1
    final_rows += rows

man2 = pd.DataFrame(final_rows)
man2.to_csv("/kaggle/working/pilot_v2/manifest.csv", index=False)

print("\n=== ORIGINALS KEPT PER FAMILY ===")
print(kept)
print("\n=== FILES ===")
print("Total files:", len(man2), "| on disk:", len(glob.glob(os.path.join(OUT, "*.pdf"))))
print(man2.groupby(["variant", "label"]).size().to_string())
print("\nSame originals as v1:", int(man2.orig_file.isin(v1.orig_file).sum() // 15), "of", man2.orig_file.nunique())
print("\n=== SPLIT (originals) ===")
print(man2.drop_duplicates("orig_file")["split"].value_counts().to_string())
print("\n=== WHY ORIGINALS WERE SKIPPED ===")
print(pd.Series(errors).value_counts().head(10).to_string() if errors else "none")

# ---------- quality check: do the forgeries really change the page text? ----------
def page_text(path, i):
    with fitz.open(path) as d:
        return d[int(i)].get_text() if int(i) < len(d) else ""

qa_rng = random.Random(0)
sample = qa_rng.sample(sorted(man2.orig_file.unique()), min(60, man2.orig_file.nunique()))
pairs = [("F1_digit_resave", "G1_resave"), ("F2_digit_recompress", "G2_recompress"), ("F3_digit_incr", "G3_note_incr"),
         ("F4_value_resave", "G1_resave"), ("F5_cover_replace", "G1_resave"), ("F6_digit_linearize", "G4_linearize")]
qa = {f: [0, 0] for f, _ in pairs}
for o in sample:
    sub = man2[man2.orig_file == o].set_index("variant")
    for f, g in pairs:
        try:
            changed = page_text(os.path.join(OUT, sub.loc[f, "out_file"]), sub.loc[f, "edit_page"]) != \
                      page_text(os.path.join(OUT, sub.loc[g, "out_file"]), sub.loc[f, "edit_page"])
            qa[f][0] += int(changed)
            qa[f][1] += 1
        except Exception:
            pass
print("\n=== QUALITY CHECK: page text actually changed vs its honest twin (sample of", len(sample), "documents) ===")
for f, (c, n) in qa.items():
    print(f"{f:22s}: {c}/{n} changed ({round(100 * c / n, 1) if n else 0}%)")

In [ ]:
# CELL 10 — Feature extraction for dataset v2 (same 54 features as CELL 7 + new L5 page-resource features)
!pip install -q pikepdf pebble

import os, re, glob, zlib, math, logging, collections
import numpy as np
import pandas as pd
import pikepdf
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pikepdf").setLevel(logging.ERROR)

MANIFEST = "/kaggle/working/pilot_v2/manifest.csv"
FILES = "/kaggle/working/pilot_v2/files"
OUT_CSV = "/kaggle/working/pilot_v2_features.csv"
assert os.path.exists(MANIFEST), "v2 manifest not found - did CELL 9 finish in this session?"
man = pd.read_csv(MANIFEST)
missing = [f for f in man.out_file if not os.path.exists(os.path.join(FILES, f))]
print("Manifest rows:", len(man), "| missing files:", len(missing))
assert not missing, f"Missing files, e.g. {missing[:3]}"

MAX_STREAMS = 120
TIME_LIMIT = 60
TOOL_WORDS = [b"pypdf", b"pymupdf", b"mupdf", b"pikepdf", b"qpdf"]
STREAM_BODY = re.compile(rb"(?<!end)stream\r?\n.*?endstream", re.S)
STD14 = {"Helvetica", "Helvetica-Bold", "Helvetica-Oblique", "Helvetica-BoldOblique", "Times-Roman", "Times-Bold",
         "Times-Italic", "Times-BoldItalic", "Courier", "Courier-Bold", "Courier-Oblique", "Courier-BoldOblique",
         "Symbol", "ZapfDingbats"}

def ratio(a, b):
    return a / b if b else 0.0

# ---------- L1: file-level handwriting (identical to CELL 7) ----------
def file_features(raw):
    f = {}
    skel = STREAM_BODY.sub(b"stream endstream", raw)
    f["size_log"] = math.log10(len(raw) + 1)
    m = re.match(rb"\s*%PDF-(\d)\.(\d)", raw[:64])
    f["version"] = int(m.group(1)) + int(m.group(2)) / 10 if m else 0
    f["binary_comment"] = int(bool(re.match(rb"\s*%PDF-\d\.\d\s*[\r\n]+%[\x80-\xff]{4}", raw[:40])))
    f["n_eof"] = raw.count(b"%%EOF")
    f["n_startxref"] = raw.count(b"startxref")
    f["xref_table"] = int(bool(re.search(rb"(^|[\r\n])xref\s*[\r\n]", skel)))
    f["xref_stream"] = int(bool(re.search(rb"/Type\s*/XRef", skel)))
    f["obj_streams"] = int(bool(re.search(rb"/Type\s*/ObjStm", skel)))
    f["linearized"] = int(b"/Linearized" in raw[:2048])
    f["true_revisions"] = f["n_eof"] - f["linearized"]
    crlf = skel.count(b"\r\n")
    cr = skel.count(b"\r") - crlf
    lf = skel.count(b"\n") - crlf
    tot = crlf + cr + lf
    f["eol_crlf"], f["eol_cr"], f["eol_lf"] = ratio(crlf, tot), ratio(cr, tot), ratio(lf, tot)
    objs = re.findall(rb"(\d+)\s+(\d+)\s+obj", skel)
    nums = [int(a) for a, _ in objs]
    f["n_obj_defs"] = len(nums)
    f["obj_order_increasing"] = ratio(sum(1 for a, b in zip(nums, nums[1:]) if b > a), max(len(nums) - 1, 1))
    f["obj_gen_nonzero"] = ratio(sum(1 for _, g in objs if g != b"0"), len(objs))
    f["obj_max_over_count"] = ratio(max(nums) if nums else 0, len(set(nums)))
    f["obj_redefined"] = len(nums) - len(set(nums))
    f["obj_newline_dict"] = ratio(len(re.findall(rb"obj\s*[\r\n]+\s*<<", skel)), len(nums))
    f["dict_open_space"] = ratio(skel.count(b"<< /"), skel.count(b"<<"))
    f["dict_close_space"] = ratio(skel.count(b" >>"), skel.count(b">>"))
    nstream = len(re.findall(rb"(?<!end)stream", skel))
    f["stream_kw_crlf"] = ratio(len(re.findall(rb"(?<!end)stream\r\n", raw)), nstream)
    f["endobj_newline_before"] = ratio(len(re.findall(rb"[\r\n]endobj", skel)), skel.count(b"endobj"))
    f["reals_dot_zero"] = ratio(len(re.findall(rb"\s\d+\.0[\s/\]>]", skel)), len(re.findall(rb"\s\d+\.\d+[\s/\]>]", skel)))
    f["has_ID"] = int(b"/ID" in skel)
    f["tool_word_in_file"] = int(any(w in skel.lower() for w in TOOL_WORDS))
    return f

# ---------- L2: per-stream compression fingerprint (identical to CELL 7) ----------
def stream_sig(s):
    d = {}
    flt = s.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    d["filter"] = "+".join(names) if names else "none"
    d["has_parms"] = int(s.get("/DecodeParms") is not None)
    d["len_indirect"] = int(bool(getattr(s.get("/Length"), "is_indirect", False)))
    d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["ratio"] = -1, 0, 0, 0, 0, 0.0
    raw = s.read_raw_bytes()
    if names == ["/FlateDecode"] and len(raw) >= 6:
        d["hdr"] = raw[0] * 256 + raw[1]
        try:
            dob = zlib.decompressobj()
            dec = dob.decompress(raw)
            body = raw[:len(raw) - len(dob.unused_data)]
            d["tail_junk"] = int(len(dob.unused_data) > 0)
            d["ratio"] = ratio(len(body), len(dec))
            d["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
            d["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
            if len(dec) <= 2_000_000:
                for lv in range(1, 10):
                    if zlib.compress(dec, lv) == body:
                        d["zlevel"] = lv
                        break
        except Exception:
            d["hdr"] = -2
    return d

# ---------- L5 (new): page resources - fonts, images, annotations, extra content streams ----------
def resource_features(pdf):
    f = {"l5_n_annots": 0, "l5_n_freetext": 0, "l5_n_text_notes": 0, "l5_n_img_xobj": 0, "l5_n_fonts": 0,
         "l5_n_std14_unembedded": 0, "l5_frac_pages_multi_stream": 0.0, "l5_max_streams_per_page": 0}
    fonts_seen, imgs_seen, multi = set(), set(), 0
    for page in pdf.pages:
        po = page.obj
        annots = po.get("/Annots")
        if isinstance(annots, pikepdf.Array):
            for a in annots:
                try:
                    st = str(a.get("/Subtype", ""))
                except Exception:
                    continue
                f["l5_n_annots"] += 1
                f["l5_n_freetext"] += int(st == "/FreeText")
                f["l5_n_text_notes"] += int(st == "/Text")
        c = po.get("/Contents")
        ns = len(c) if isinstance(c, pikepdf.Array) else (1 if c is not None else 0)
        multi += int(ns > 1)
        f["l5_max_streams_per_page"] = max(f["l5_max_streams_per_page"], ns)
        try:
            res = page.resources
        except Exception:
            res = po.get("/Resources")
        if res is None:
            continue
        fdict = res.get("/Font")
        if isinstance(fdict, pikepdf.Dictionary):
            for _, fo in fdict.items():
                key = getattr(fo, "objgen", (0, 0))
                key = key if key != (0, 0) else id(fo)
                if key in fonts_seen:
                    continue
                fonts_seen.add(key)
                base = str(fo.get("/BaseFont", "")).lstrip("/")
                desc = fo.get("/FontDescriptor")
                embedded = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                if base in STD14 and not embedded and str(fo.get("/Subtype", "")) != "/Type0":
                    f["l5_n_std14_unembedded"] += 1
        xdict = res.get("/XObject")
        if isinstance(xdict, pikepdf.Dictionary):
            for _, xo in xdict.items():
                key = getattr(xo, "objgen", (0, 0))
                if key in imgs_seen:
                    continue
                try:
                    if str(xo.get("/Subtype", "")) == "/Image":
                        imgs_seen.add(key)
                except Exception:
                    pass
    f["l5_n_fonts"] = len(fonts_seen)
    f["l5_n_img_xobj"] = len(imgs_seen)
    f["l5_frac_pages_multi_stream"] = ratio(multi, len(pdf.pages))
    return f

def extract(name):
    path = os.path.join(FILES, name)
    with open(path, "rb") as fh:
        raw = fh.read()
    f = {"out_file": name}
    f.update(file_features(raw))
    with pikepdf.open(path) as pdf:
        info = pdf.docinfo
        prod = str(info.get("/Producer", "")) if info is not None else ""
        f["has_producer"] = int(bool(prod.strip()))
        f["producer_len"] = len(prod)
        cd, md = str(info.get("/CreationDate", "")), str(info.get("/ModDate", ""))
        f["moddate_eq_creation"] = int(bool(cd) and cd == md)
        f["has_xmp"] = int("/Metadata" in pdf.Root)
        f["n_pages"] = len(pdf.pages)
        try:
            f.update(resource_features(pdf))
        except Exception:
            pass
        streams = []
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream):
                    streams.append((pi, s))
            if len(streams) >= MAX_STREAMS:
                break
        streams = streams[:MAX_STREAMS]
        sigs = []
        for pi, s in streams:
            try:
                sigs.append((pi, getattr(s, "objgen", (-1, 0))[0], stream_sig(s)))
            except Exception:
                pass
        cs_nums = {num for _, num, _ in sigs}

    # ---------- L3: consistency between page streams (identical to CELL 7) ----------
    n = len(sigs)
    f["cs_n"] = n
    if n:
        fl = [d["filter"] for _, _, d in sigs]
        f["cs_frac_flate"] = ratio(fl.count("/FlateDecode"), n)
        f["cs_frac_lzw"] = ratio(sum("LZW" in x for x in fl), n)
        f["cs_frac_none"] = ratio(fl.count("none"), n)
        f["cs_n_filters"] = len(set(fl))
        flate = [d for _, _, d in sigs if d["filter"] == "/FlateDecode"]
        f["cs_frac_zlib_repro"] = ratio(sum(d["zlevel"] > 0 for d in flate), len(flate)) if flate else np.nan
        f["cs_n_hdr"] = len({d["hdr"] for _, _, d in sigs})
        f["cs_n_zlevel"] = len({d["zlevel"] for _, _, d in sigs})
        f["cs_frac_sync"] = np.mean([d["sync_end"] for _, _, d in sigs])
        f["cs_frac_adler"] = np.mean([d["adler_ok"] for _, _, d in sigs])
        f["cs_frac_junk"] = np.mean([d["tail_junk"] for _, _, d in sigs])
        f["cs_frac_len_indirect"] = np.mean([d["len_indirect"] for _, _, d in sigs])
        r = np.array([d["ratio"] for _, _, d in sigs])
        f["cs_ratio_std"] = float(r.std())
        f["cs_ratio_max_z"] = float(np.max(np.abs(r - r.mean())) / r.std()) if r.std() > 0 else 0.0
        keys = [(d["filter"], d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["has_parms"], d["len_indirect"]) for _, _, d in sigs]
        cnt = collections.Counter(keys)
        f["cs_n_sig"] = len(cnt)
        f["cs_majority_share"] = max(cnt.values()) / n
        f["cs_n_singletons"] = sum(1 for v in cnt.values() if v == 1) if n >= 3 else 0
        odd = [pi for (pi, _, _), k in zip(sigs, keys) if cnt[k] == 1] if n >= 3 else []
        f["loc_page"] = odd[0] if len(odd) == 1 else -1

    # ---------- L4: what the last update touched (identical to CELL 7) ----------
    f.update({"upd_n_objs": 0, "upd_touch_content": 0, "upd_has_annot": 0, "upd_has_page": 0, "upd_has_sig": 0})
    if f["true_revisions"] > 1:
        ends = [m.end() for m in re.finditer(rb"%%EOF", raw)]
        tail = STREAM_BODY.sub(b"stream endstream", raw[ends[-2]:])
        objs = re.findall(rb"(\d+)\s+\d+\s+obj(.*?)endobj", tail, re.S)
        f["upd_n_objs"] = len(objs)
        for num, body in objs:
            if int(num) in cs_nums:
                f["upd_touch_content"] = 1
            if re.search(rb"/Type\s*/Annot|/Subtype\s*/(Text|FreeText)\b", body):
                f["upd_has_annot"] = 1
            if re.search(rb"/Type\s*/Page\b", body):
                f["upd_has_page"] = 1
            if b"/ByteRange" in body:
                f["upd_has_sig"] = 1
    return f

rows, errors = [], []
with ProcessPool(max_workers=4) as pool:
    futs = {pool.schedule(extract, args=(n,), timeout=TIME_LIMIT): n for n in man.out_file}
    for i, fu in enumerate(tqdm(as_completed(futs), total=len(futs), desc="Files")):
        name = futs[fu]
        try:
            rows.append(fu.result())
        except TimeoutError:
            errors.append((name, "Timeout"))
        except Exception as ex:
            errors.append((name, type(ex).__name__))
        if (i + 1) % 500 == 0:
            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

feat = man.merge(pd.DataFrame(rows), on="out_file", how="left")
feat.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV, "| rows:", len(feat), "| feature columns:", feat.shape[1] - man.shape[1])
print("Files that failed:", len(errors), pd.Series([e for _, e in errors]).value_counts().to_dict() if errors else "")

g0 = feat[feat.variant == "G0_original"]
print("\n=== CHECK: share of original Flate page streams exactly reproducible by Python zlib, by family ===")
print(g0.groupby("family")["cs_frac_zlib_repro"].mean().mul(100).round(1).to_string())

cols = ["true_revisions", "cs_n_hdr", "cs_n_sig", "cs_n_singletons", "cs_frac_zlib_repro",
        "upd_touch_content", "upd_has_annot", "l5_n_freetext", "l5_n_img_xobj",
        "l5_n_std14_unembedded", "l5_frac_pages_multi_stream"]
print("\n=== PREVIEW: average of key features per version ===")
print(feat.groupby("variant")[cols].mean().round(3).to_string())

dig = feat[feat.variant.isin(["F1_digit_resave", "F2_digit_recompress", "F6_digit_linearize"]) & feat.edit_page.notna()]
hit = dig[dig.loc_page >= 0]
print("\n=== PREVIEW: localization on digit forgeries (odd-one-out stream -> edited page?) ===")
print("Forgeries where exactly one stream stood out:", len(hit), "of", len(dig))
print("Of those, correct page:", round((hit.loc_page == hit.edit_page).mean() * 100, 1) if len(hit) else 0, "%")

In [ ]:
# CELL 11 — v2 experiments: baselines vs our model, per forgery type, per twin pair, per tool/family, LOTO, v1->v2, localization, CIs
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")

df = pd.read_csv("/kaggle/working/pilot_v2_features.csv")
print("Rows:", len(df), "| train:", int((df.split == "train").sum()), "| test:", int((df.split == "test").sum()),
      "| test documents:", df[df.split == "test"].orig_file.nunique())

META = ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"]
L1 = ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream", "obj_streams",
      "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs", "obj_order_increasing",
      "obj_gen_nonzero", "obj_max_over_count", "obj_redefined", "obj_newline_dict", "dict_open_space",
      "dict_close_space", "stream_kw_crlf", "endobj_newline_before", "reals_dot_zero", "has_ID",
      "tool_word_in_file", "n_pages"]
L23 = ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters", "cs_frac_zlib_repro", "cs_n_hdr",
       "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk", "cs_frac_len_indirect", "cs_ratio_std",
       "cs_ratio_max_z", "cs_n_sig", "cs_majority_share", "cs_n_singletons"]
L4 = ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"]
L5 = ["l5_n_annots", "l5_n_freetext", "l5_n_text_notes", "l5_n_img_xobj", "l5_n_fonts", "l5_n_std14_unembedded",
      "l5_frac_pages_multi_stream", "l5_max_streams_per_page"]
V1SET = sorted(set(META + L1 + L23 + L4))
FULL = sorted(set(V1SET + L5))
missing = [c for c in FULL if c not in df.columns]
assert not missing, f"Missing feature columns: {missing}"

GROUPS = {
    "META (metadata only)": META,
    "L1 (file style)": L1,
    "L2+L3 (compression + consistency)": L23,
    "L4 (last-update diff)": L4,
    "L5 (page resources, new)": L5,
    "v1 feature set (L1-L4)": V1SET,
    "FULL MODEL (L1-L5)": FULL,
}
PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

def tpr_at(y, s, fpr):
    thr = np.quantile(s[y == 0], 1 - fpr)
    return float((s[y == 1] > thr).mean())

def auc_or_nan(y, s):
    try:
        return round(roc_auc_score(y, s), 3)
    except ValueError:
        return np.nan

def scores(y, s):
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    return {"ROC-AUC": auc_or_nan(y, s), "PR-AUC": round(average_precision_score(y, s), 3),
            "Detect@1%FA": round(tpr_at(y, s, 0.01), 3), "Detect@5%FA": round(tpr_at(y, s, 0.05), 3)}

def fit(cols, train):
    return lgb.LGBMClassifier(**PARAMS).fit(train[cols], train["label"])

tr = df[df.split == "train"].reset_index(drop=True)
te = df[df.split == "test"].reset_index(drop=True)

NO_EVIDENCE_F = ["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake"]
NO_EVIDENCE_G = ["G2_recompress", "G5_stamp_honest", "G6_textbox_honest"]
EVID = ~te.variant.isin(NO_EVIDENCE_F + NO_EVIDENCE_G)

# ---------- E1: main results ----------
rows, models = [], {}
rows.append({"Method": "Rule: %%EOF count > 1 (industry check)", **scores(te.label, te.n_eof)})
rows.append({"Method": "Rule: true revisions > 1", **scores(te.label, te.true_revisions)})
for g, cols in GROUPS.items():
    m = fit(cols, tr)
    models[g] = m
    rows.append({"Method": f"LightGBM: {g}", **scores(te.label, m.predict_proba(te[cols])[:, 1])})
print("\n=== E1: MAIN RESULTS, all 15 versions (test set, unseen documents) ===")
print(pd.DataFrame(rows).to_string(index=False))

te["score"] = models["FULL MODEL (L1-L5)"].predict_proba(te[FULL])[:, 1]
rows2 = [{"Method": "Rule: %%EOF count > 1", **scores(te.label[EVID], te.n_eof[EVID])},
         {"Method": "LightGBM: FULL MODEL (L1-L5)", **scores(te.label[EVID], te.score[EVID])}]
print("\n=== E1b: ONLY forgeries that leave structural evidence (excl. F2/F7/F8 and their twins G2/G5/G6) ===")
print(pd.DataFrame(rows2).to_string(index=False))

# ---------- threshold chosen on TRAIN only (~5% false alarms, out-of-fold, grouped by document) ----------
oof = np.zeros(len(tr))
for a, b in GroupKFold(n_splits=5).split(tr, tr.label, tr.orig_file):
    oof[b] = fit(FULL, tr.iloc[a]).predict_proba(tr.iloc[b][FULL])[:, 1]
THR = float(np.quantile(oof[tr.label.values == 0], 0.95))
te["flagged"] = (te.score > THR).astype(int)
print(f"\nThreshold (set on training data for ~5% false alarms): {THR:.3f}")

# ---------- E2: per version ----------
print("\n=== E2: FLAG RATE PER VERSION (G = false alarms, F = detections) ===")
pv = te.groupby(["variant", "label"])["flagged"].mean().mul(100).round(1).reset_index()
pv.columns = ["variant", "label", "flagged_%"]
print(pv.to_string(index=False))

# ---------- E2b: per twin pair ----------
print("\n=== E2b: SEPARATION PER TWIN PAIR (ROC-AUC) ===")
pairs = [("G1_resave", "F1_digit_resave"), ("G2_recompress", "F2_digit_recompress"), ("G3_note_incr", "F3_digit_incr"),
         ("G1_resave", "F4_value_resave"), ("G1_resave", "F5_cover_replace"), ("G4_linearize", "F6_digit_linearize"),
         ("G5_stamp_honest", "F7_stamp_fake"), ("G6_textbox_honest", "F8_textbox_fake")]
for g, f in pairs:
    sub = te[te.variant.isin([g, f])]
    sub0 = te[te.variant.isin(["G0_original", f])]
    print(f"{f:20s} vs twin {g:18s}: AUC = {auc_or_nan(sub.label, sub.score):<6} | vs untouched original: {auc_or_nan(sub0.label, sub0.score)}")

# ---------- E3: per tool ----------
print("\n=== E3: PER TOOL (ROC-AUC vs twin) ===")
for t in ["pikepdf", "pymupdf", "pypdf"]:
    out = []
    for g, f in [("G1_resave", "F1_digit_resave"), ("G1_resave", "F4_value_resave"), ("G2_recompress", "F2_digit_recompress")]:
        sub = te[(te.tool == t) & te.variant.isin([g, f])]
        out.append(f"{f.split('_')[0]}: {auc_or_nan(sub.label, sub.score)} (n={len(sub)})")
    print(f"{t:8s} | " + " | ".join(out))

# ---------- E4: per family ----------
print("\n=== E4: PER ISSUER FAMILY (ROC-AUC) ===")
for fam in sorted(te.family.unique()):
    s_all = te[te.family == fam]
    s_ev = te[(te.family == fam) & EVID]
    print(f"{fam:28s} | all versions: {auc_or_nan(s_all.label, s_all.score)} | evidence-bearing only: {auc_or_nan(s_ev.label, s_ev.score)}")

# ---------- E5: leave-one-tool-out ----------
print("\n=== E5: LEAVE-ONE-TOOL-OUT (train without the tool, test only on it) ===")
RW_TR = ["G0_original", "G1_resave", "G2_recompress", "F1_digit_resave", "F2_digit_recompress", "F4_value_resave"]
for t in ["pikepdf", "pymupdf", "pypdf"]:
    m = fit(FULL, tr[tr.variant.isin(RW_TR) & (tr.tool != t)])
    te_all = te[te.variant.isin(RW_TR[1:]) & (te.tool == t)]
    te_ev = te[te.variant.isin(["G1_resave", "F1_digit_resave", "F4_value_resave"]) & (te.tool == t)]
    print(f"unseen tool = {t:8s}: AUC incl. laundering = {auc_or_nan(te_all.label, m.predict_proba(te_all[FULL])[:, 1])}"
          f" | resave forgeries only = {auc_or_nan(te_ev.label, m.predict_proba(te_ev[FULL])[:, 1])}")

# ---------- E6: v1 model -> v2 data (did v1 learn a generator artifact?) ----------
V1_CSV = "/kaggle/working/pilot_features.csv"
print("\n=== E6: TRAIN ON v1 (random compression levels), TEST ON v2 (tool-native compression) ===")
if os.path.exists(V1_CSV):
    v1 = pd.read_csv(V1_CSV)
    m1 = fit(V1SET, v1[v1.split == "train"])
    for g, f in [("G1_resave", "F1_digit_resave"), ("G0_original", "F1_digit_resave"), ("G1_resave", "F4_value_resave"),
                 ("G4_linearize", "F6_digit_linearize")]:
        sub = te[te.variant.isin([g, f])]
        print(f"v1-trained model on v2 {f:20s} vs {g:15s}: AUC = {auc_or_nan(sub.label, m1.predict_proba(sub[V1SET])[:, 1])}")
else:
    print("v1 features file not in this session - skipped (not essential)")

# ---------- E7: localization (rule-based, no training) ----------
print("\n=== E7: LOCALIZATION - does the odd-one-out stream point to the edited page? (all documents) ===")
for v in ["F1_digit_resave", "F4_value_resave", "F5_cover_replace", "F6_digit_linearize", "F2_digit_recompress"]:
    sub = df[(df.variant == v) & df.edit_page.notna()]
    cov = sub[sub.loc_page >= 0]
    acc = round((cov.loc_page == cov.edit_page).mean() * 100, 1) if len(cov) else 0
    print(f"{v:20s}: one stream stood out in {round(len(cov) / max(len(sub), 1) * 100, 1)}% | correct page in {acc}% of those")

# ---------- E8: feature importance + ablations ----------
imp = pd.Series(models["FULL MODEL (L1-L5)"].booster_.feature_importance("gain"), index=FULL)
print("\n=== E8: TOP 15 FEATURES (full model, % of gain) ===")
print((imp / imp.sum() * 100).sort_values(ascending=False).head(15).round(1).to_string())
print("\n=== E8b: ABLATIONS (evidence-bearing test rows, ROC-AUC) ===")
for name, cols in [("FULL", FULL), ("FULL without cs_n_hdr", [c for c in FULL if c != "cs_n_hdr"]),
                   ("FULL without all compression features (L2+L3)", [c for c in FULL if c not in L23]),
                   ("FULL without L5", V1SET)]:
    m = fit(cols, tr)
    print(f"{name:48s}: {auc_or_nan(te.label[EVID], m.predict_proba(te[EVID][cols])[:, 1])}")

# ---------- 95% confidence intervals (bootstrap over test documents) ----------
rng = np.random.default_rng(0)
docs = te.orig_file.unique()
by_doc = {d: te.index[te.orig_file == d].values for d in docs}
aucs_all, aucs_ev = [], []
for _ in range(300):
    idx = np.concatenate([by_doc[d] for d in rng.choice(docs, size=len(docs), replace=True)])
    s = te.loc[idx]
    aucs_all.append(roc_auc_score(s.label, s.score))
    se = s[~s.variant.isin(NO_EVIDENCE_F + NO_EVIDENCE_G)]
    aucs_ev.append(roc_auc_score(se.label, se.score))
print("\n=== 95% CONFIDENCE INTERVALS (FULL model ROC-AUC, bootstrap over test documents) ===")
print(f"All versions:           {np.mean(aucs_all):.3f}  [{np.percentile(aucs_all, 2.5):.3f}, {np.percentile(aucs_all, 97.5):.3f}]")
print(f"Evidence-bearing only:  {np.mean(aucs_ev):.3f}  [{np.percentile(aucs_ev, 2.5):.3f}, {np.percentile(aucs_ev, 97.5):.3f}]")

te.to_csv("/kaggle/working/pilot_v2_test_predictions.csv", index=False)
print("\nSaved: /kaggle/working/pilot_v2_test_predictions.csv")

In [7]:
# CELL 12 — Convert every v2 PDF into a graph (nodes = PDF objects, edges = references) and save one compact file
!pip install -q pikepdf pebble

import os, re, zlib, math, pickle, gzip, logging
import numpy as np
import pandas as pd
import pikepdf
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pikepdf").setLevel(logging.ERROR)

MANIFEST = "/kaggle/working/pilot_v2/manifest.csv"
FILES = "/kaggle/working/pilot_v2/files"
OUT_GRAPHS = "/kaggle/working/graphs_v2.pkl.gz"
assert os.path.exists(MANIFEST), "v2 manifest not found in this session"
man = pd.read_csv(MANIFEST)
missing = [f for f in man.out_file if not os.path.exists(os.path.join(FILES, f))]
print("Files:", len(man), "| missing:", len(missing))
assert not missing

TIME_LIMIT = 90
NODE_TYPES = ["catalog", "pages", "page", "content", "font", "fontdesc", "fontfile", "image", "form", "annot",
              "metadata", "xref", "extgstate", "other_dict", "array", "other_stream", "other"]
EDGE_KEYS = ["/Parent", "/Kids", "/Contents", "/Resources", "/Font", "/XObject", "/Annots", "/FontDescriptor",
             "/FontFile", "/Metadata", "/Pages", "other"]
STD14 = {"Helvetica", "Helvetica-Bold", "Helvetica-Oblique", "Helvetica-BoldOblique", "Times-Roman", "Times-Bold",
         "Times-Italic", "Times-BoldItalic", "Courier", "Courier-Bold", "Courier-Oblique", "Courier-BoldOblique",
         "Symbol", "ZapfDingbats"}
HDRS = [0x789C, 0x78DA, 0x7801, 0x785E]
FEAT_NAMES = (["t_" + t for t in NODE_TYPES] + ["is_stream", "raw_len_log", "n_keys_log"] +
              ["f_flate", "f_lzw", "f_dct", "f_none", "f_other"] +
              ["h_789C", "h_78DA", "h_7801", "h_785E", "h_other"] +
              ["zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio",
               "objnum_rank", "gen_nonzero", "std14_unembedded", "annot_freetext", "annot_text"])
D = len(FEAT_NAMES)

def edge_key(k):
    if k is None:
        return "other"
    if k.startswith("/FontFile"):
        return "/FontFile"
    return k if k in EDGE_KEYS else "other"

def collect_refs(o, carried, out, depth=0):
    if depth > 8:
        return
    try:
        if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
            for k, v in o.items():
                kk = k if (k in EDGE_KEYS or k.startswith("/FontFile")) else carried
                if v.is_indirect:
                    out.append((edge_key(kk), v.objgen))
                else:
                    collect_refs(v, kk, out, depth + 1)
        elif isinstance(o, pikepdf.Array):
            for v in o:
                if v.is_indirect:
                    out.append((edge_key(carried), v.objgen))
                else:
                    collect_refs(v, carried, out, depth + 1)
    except Exception:
        pass

def node_type(o, content_set):
    if isinstance(o, pikepdf.Stream):
        if o.objgen in content_set:
            return "content"
        st, ty = str(o.get("/Subtype", "")), str(o.get("/Type", ""))
        if st == "/Image":
            return "image"
        if st == "/Form":
            return "form"
        if ty == "/Metadata":
            return "metadata"
        if ty in ("/XRef", "/ObjStm"):
            return "xref"
        if "/Length1" in o or st in ("/Type1C", "/CIDFontType0C", "/OpenType"):
            return "fontfile"
        return "other_stream"
    if isinstance(o, pikepdf.Dictionary):
        ty, st = str(o.get("/Type", "")), str(o.get("/Subtype", ""))
        mapping = {"/Catalog": "catalog", "/Pages": "pages", "/Page": "page", "/Font": "font",
                   "/FontDescriptor": "fontdesc", "/Annot": "annot", "/ExtGState": "extgstate"}
        if ty in mapping:
            return mapping[ty]
        if st in ("/Text", "/FreeText", "/Link", "/Widget", "/Stamp", "/Square", "/Highlight"):
            return "annot"
        return "other_dict"
    if isinstance(o, pikepdf.Array):
        return "array"
    return "other"

def stream_feats(o, is_content):
    v = {"raw_len_log": 0.0, "f": "f_none", "h": None, "zlevel": 0, "zlib_repro": 0, "adler_ok": 0,
         "sync_end": 0, "tail_junk": 0, "ratio": 0.0}
    flt = o.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    raw = o.read_raw_bytes()
    v["raw_len_log"] = math.log10(len(raw) + 1) / 7
    if not names:
        v["f"] = "f_none"
    elif names == ["/FlateDecode"]:
        v["f"] = "f_flate"
    elif any("LZW" in n for n in names):
        v["f"] = "f_lzw"
    elif any("DCT" in n for n in names):
        v["f"] = "f_dct"
    else:
        v["f"] = "f_other"
    if v["f"] == "f_flate" and len(raw) >= 6:
        hdr = raw[0] * 256 + raw[1]
        v["h"] = "h_%04X" % hdr if hdr in HDRS else "h_other"
        if is_content or len(raw) <= 200_000:
            try:
                dob = zlib.decompressobj()
                dec = dob.decompress(raw)
                body = raw[:len(raw) - len(dob.unused_data)]
                v["tail_junk"] = int(len(dob.unused_data) > 0)
                v["ratio"] = min(len(body) / max(len(dec), 1), 1.5)
                v["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
                v["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
                if len(dec) <= 2_000_000:
                    for lv in range(1, 10):
                        if zlib.compress(dec, lv) == body:
                            v["zlevel"], v["zlib_repro"] = lv, 1
                            break
            except Exception:
                pass
    return v

def to_graph(name):
    path = os.path.join(FILES, name)
    with pikepdf.open(path) as pdf:
        content_page = {}
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream) and s.objgen not in content_page:
                    content_page[s.objgen] = pi
        objs = [o for o in pdf.objects if o.objgen != (0, 0)]
        idx = {o.objgen: i for i, o in enumerate(objs)}
        max_num = max([g[0] for g in idx] + [1])
        X = np.zeros((len(objs), D), dtype=np.float16)
        node_page = np.full(len(objs), -1, dtype=np.int16)
        col = {n: j for j, n in enumerate(FEAT_NAMES)}
        src, dst, et = [], [], []
        for i, o in enumerate(objs):
            t = node_type(o, content_page)
            X[i, col["t_" + t]] = 1
            X[i, col["objnum_rank"]] = o.objgen[0] / max_num
            X[i, col["gen_nonzero"]] = int(o.objgen[1] != 0)
            if o.objgen in content_page:
                node_page[i] = content_page[o.objgen]
            try:
                if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
                    X[i, col["n_keys_log"]] = math.log10(len(list(o.keys())) + 1)
                if isinstance(o, pikepdf.Stream):
                    X[i, col["is_stream"]] = 1
                    sf = stream_feats(o, t == "content")
                    X[i, col["raw_len_log"]] = sf["raw_len_log"]
                    X[i, col[sf["f"]]] = 1
                    if sf["h"]:
                        X[i, col[sf["h"]]] = 1
                    X[i, col["zlevel"]] = sf["zlevel"] / 9
                    for k in ["zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio"]:
                        X[i, col[k]] = sf[k]
                if t == "font":
                    base = str(o.get("/BaseFont", "")).lstrip("/")
                    desc = o.get("/FontDescriptor")
                    emb = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                    X[i, col["std14_unembedded"]] = int(base in STD14 and not emb and str(o.get("/Subtype", "")) != "/Type0")
                if t == "annot":
                    st = str(o.get("/Subtype", ""))
                    X[i, col["annot_freetext"]] = int(st == "/FreeText")
                    X[i, col["annot_text"]] = int(st == "/Text")
            except Exception:
                pass
            refs = []
            collect_refs(o, None, refs)
            for k, tgt in refs:
                if tgt in idx:
                    src.append(i)
                    dst.append(idx[tgt])
                    et.append(EDGE_KEYS.index(k))
    return {"out_file": name, "x": X, "edge_index": np.array([src, dst], dtype=np.int32),
            "edge_type": np.array(et, dtype=np.int8), "node_page": node_page}

graphs, errors = [], []
with ProcessPool(max_workers=4) as pool:
    futs = {pool.schedule(to_graph, args=(n,), timeout=TIME_LIMIT): n for n in man.out_file}
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Graphs"):
        name = futs[fu]
        try:
            graphs.append(fu.result())
        except TimeoutError:
            errors.append((name, "Timeout"))
        except Exception as ex:
            errors.append((name, type(ex).__name__))

meta_cols = ["out_file", "orig_file", "family", "variant", "label", "tool", "save_mode", "edit_type",
             "edit_page", "twin", "split"]
meta = man[meta_cols].set_index("out_file")
for g in graphs:
    g.update(meta.loc[g["out_file"]].to_dict())

with gzip.open(OUT_GRAPHS, "wb") as fh:
    pickle.dump({"graphs": graphs, "feat_names": FEAT_NAMES, "node_types": NODE_TYPES, "edge_keys": EDGE_KEYS},
                fh, protocol=pickle.HIGHEST_PROTOCOL)

n_nodes = np.array([g["x"].shape[0] for g in graphs])
n_edges = np.array([g["edge_index"].shape[1] for g in graphs])
print("\nSaved:", OUT_GRAPHS, "| size:", round(os.path.getsize(OUT_GRAPHS) / 1e6, 1), "MB")
print("Graphs:", len(graphs), "| failed:", len(errors), pd.Series([e for _, e in errors]).value_counts().to_dict() if errors else "")
print("Nodes per graph: median", int(np.median(n_nodes)), "| 95th pct", int(np.percentile(n_nodes, 95)), "| max", int(n_nodes.max()))
print("Edges per graph: median", int(np.median(n_edges)), "| 95th pct", int(np.percentile(n_edges, 95)), "| max", int(n_edges.max()))
print("Node features per node:", D, "| edge types:", len(EDGE_KEYS))
print("\nGraphs per version:")
print(pd.Series([g["variant"] for g in graphs]).value_counts().sort_index().to_string())

Files: 5835 | missing: 0


Graphs:   0%|          | 0/5835 [00:00<?, ?it/s]


Saved: /kaggle/working/graphs_v2.pkl.gz | size: 9.0 MB
Graphs: 4190 | failed: 1645 {'AttributeError': 1645}
Nodes per graph: median 87 | 95th pct 1599 | max 12644
Edges per graph: median 132 | 95th pct 2207 | max 21169
Node features per node: 41 | edge types: 12

Graphs per version:
F1_digit_resave        320
F2_digit_recompress    334
F3_digit_incr          226
F4_value_resave        320
F5_cover_replace       226
F6_digit_linearize     377
F7_stamp_fake          226
F8_textbox_fake        226
G0_original            226
G1_resave              320
G2_recompress          334
G3_note_incr           226
G4_linearize           377
G5_stamp_honest        226
G6_textbox_honest      226


In [8]:
# CELL 12 (fixed) — Convert every v2 PDF into a graph; skips plain-number objects, adds len_indirect + entropy node features
!pip install -q pikepdf pebble

import os, re, zlib, math, pickle, gzip, logging
import numpy as np
import pandas as pd
import pikepdf
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pikepdf").setLevel(logging.ERROR)

MANIFEST = "/kaggle/working/pilot_v2/manifest.csv"
FILES = "/kaggle/working/pilot_v2/files"
OUT_GRAPHS = "/kaggle/working/graphs_v2.pkl.gz"
assert os.path.exists(MANIFEST), "v2 manifest not found in this session"
man = pd.read_csv(MANIFEST)
missing = [f for f in man.out_file if not os.path.exists(os.path.join(FILES, f))]
print("Files:", len(man), "| missing:", len(missing))
assert not missing

TIME_LIMIT = 90
NODE_TYPES = ["catalog", "pages", "page", "content", "font", "fontdesc", "fontfile", "image", "form", "annot",
              "metadata", "xref", "extgstate", "other_dict", "array", "other_stream", "other"]
EDGE_KEYS = ["/Parent", "/Kids", "/Contents", "/Resources", "/Font", "/XObject", "/Annots", "/FontDescriptor",
             "/FontFile", "/Metadata", "/Pages", "other"]
STD14 = {"Helvetica", "Helvetica-Bold", "Helvetica-Oblique", "Helvetica-BoldOblique", "Times-Roman", "Times-Bold",
         "Times-Italic", "Times-BoldItalic", "Courier", "Courier-Bold", "Courier-Oblique", "Courier-BoldOblique",
         "Symbol", "ZapfDingbats"}
HDRS = [0x789C, 0x78DA, 0x7801, 0x785E]
FEAT_NAMES = (["t_" + t for t in NODE_TYPES] + ["is_stream", "raw_len_log", "n_keys_log"] +
              ["f_flate", "f_lzw", "f_dct", "f_none", "f_other"] +
              ["h_789C", "h_78DA", "h_7801", "h_785E", "h_other"] +
              ["zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio",
               "objnum_rank", "gen_nonzero", "std14_unembedded", "annot_freetext", "annot_text",
               "len_indirect", "entropy"])
D = len(FEAT_NAMES)
LEN_REF = re.compile(rb"/Length\s+\d+\s+\d+\s+R")

def edge_key(k):
    if k is None:
        return "other"
    if k.startswith("/FontFile"):
        return "/FontFile"
    return k if k in EDGE_KEYS else "other"

def collect_refs(o, carried, out, depth=0):
    if depth > 8:
        return
    try:
        if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
            for k, v in o.items():
                kk = k if (k in EDGE_KEYS or k.startswith("/FontFile")) else carried
                if getattr(v, "is_indirect", False):
                    out.append((edge_key(kk), v.objgen))
                else:
                    collect_refs(v, kk, out, depth + 1)
        elif isinstance(o, pikepdf.Array):
            for v in o:
                if getattr(v, "is_indirect", False):
                    out.append((edge_key(carried), v.objgen))
                else:
                    collect_refs(v, carried, out, depth + 1)
    except Exception:
        pass

def node_type(o, content_set):
    if isinstance(o, pikepdf.Stream):
        if o.objgen in content_set:
            return "content"
        st, ty = str(o.get("/Subtype", "")), str(o.get("/Type", ""))
        if st == "/Image":
            return "image"
        if st == "/Form":
            return "form"
        if ty == "/Metadata":
            return "metadata"
        if ty in ("/XRef", "/ObjStm"):
            return "xref"
        if "/Length1" in o or st in ("/Type1C", "/CIDFontType0C", "/OpenType"):
            return "fontfile"
        return "other_stream"
    if isinstance(o, pikepdf.Dictionary):
        ty, st = str(o.get("/Type", "")), str(o.get("/Subtype", ""))
        mapping = {"/Catalog": "catalog", "/Pages": "pages", "/Page": "page", "/Font": "font",
                   "/FontDescriptor": "fontdesc", "/Annot": "annot", "/ExtGState": "extgstate"}
        if ty in mapping:
            return mapping[ty]
        if st in ("/Text", "/FreeText", "/Link", "/Widget", "/Stamp", "/Square", "/Highlight"):
            return "annot"
        return "other_dict"
    if isinstance(o, pikepdf.Array):
        return "array"
    return "other"

def shannon(raw):
    if not raw:
        return 0.0
    b = np.frombuffer(raw[:2_000_000], dtype=np.uint8)
    c = np.bincount(b, minlength=256)
    p = c[c > 0] / len(b)
    return float(-(p * np.log2(p)).sum() / 8)

def stream_feats(o, is_content):
    v = {"raw_len_log": 0.0, "f": "f_none", "h": None, "zlevel": 0, "zlib_repro": 0, "adler_ok": 0,
         "sync_end": 0, "tail_junk": 0, "ratio": 0.0, "len_indirect": 0, "entropy": 0.0}
    try:
        v["len_indirect"] = int(bool(LEN_REF.search(o.stream_dict.unparse())))
    except Exception:
        pass
    flt = o.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    raw = o.read_raw_bytes()
    v["raw_len_log"] = math.log10(len(raw) + 1) / 7
    v["entropy"] = shannon(raw)
    if not names:
        v["f"] = "f_none"
    elif names == ["/FlateDecode"]:
        v["f"] = "f_flate"
    elif any("LZW" in n for n in names):
        v["f"] = "f_lzw"
    elif any("DCT" in n for n in names):
        v["f"] = "f_dct"
    else:
        v["f"] = "f_other"
    if v["f"] == "f_flate" and len(raw) >= 6:
        hdr = raw[0] * 256 + raw[1]
        v["h"] = "h_%04X" % hdr if hdr in HDRS else "h_other"
        if is_content or len(raw) <= 200_000:
            try:
                dob = zlib.decompressobj()
                dec = dob.decompress(raw)
                body = raw[:len(raw) - len(dob.unused_data)]
                v["tail_junk"] = int(len(dob.unused_data) > 0)
                v["ratio"] = min(len(body) / max(len(dec), 1), 1.5)
                v["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
                v["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
                if len(dec) <= 2_000_000:
                    for lv in range(1, 10):
                        if zlib.compress(dec, lv) == body:
                            v["zlevel"], v["zlib_repro"] = lv, 1
                            break
            except Exception:
                pass
    return v

def to_graph(name):
    path = os.path.join(FILES, name)
    with pikepdf.open(path) as pdf:
        content_page = {}
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream) and s.objgen not in content_page:
                    content_page[s.objgen] = pi
        objs = [o for o in pdf.objects if hasattr(o, "objgen") and o.objgen != (0, 0)]
        idx = {o.objgen: i for i, o in enumerate(objs)}
        max_num = max([g[0] for g in idx] + [1])
        X = np.zeros((len(objs), D), dtype=np.float16)
        node_page = np.full(len(objs), -1, dtype=np.int16)
        col = {n: j for j, n in enumerate(FEAT_NAMES)}
        src, dst, et = [], [], []
        for i, o in enumerate(objs):
            t = node_type(o, content_page)
            X[i, col["t_" + t]] = 1
            X[i, col["objnum_rank"]] = o.objgen[0] / max_num
            X[i, col["gen_nonzero"]] = int(o.objgen[1] != 0)
            if o.objgen in content_page:
                node_page[i] = content_page[o.objgen]
            try:
                if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
                    X[i, col["n_keys_log"]] = math.log10(len(list(o.keys())) + 1)
                if isinstance(o, pikepdf.Stream):
                    X[i, col["is_stream"]] = 1
                    sf = stream_feats(o, t == "content")
                    X[i, col["raw_len_log"]] = sf["raw_len_log"]
                    X[i, col[sf["f"]]] = 1
                    if sf["h"]:
                        X[i, col[sf["h"]]] = 1
                    X[i, col["zlevel"]] = sf["zlevel"] / 9
                    for k in ["zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio", "len_indirect", "entropy"]:
                        X[i, col[k]] = sf[k]
                if t == "font":
                    base = str(o.get("/BaseFont", "")).lstrip("/")
                    desc = o.get("/FontDescriptor")
                    emb = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                    X[i, col["std14_unembedded"]] = int(base in STD14 and not emb and str(o.get("/Subtype", "")) != "/Type0")
                if t == "annot":
                    st = str(o.get("/Subtype", ""))
                    X[i, col["annot_freetext"]] = int(st == "/FreeText")
                    X[i, col["annot_text"]] = int(st == "/Text")
            except Exception:
                pass
            refs = []
            collect_refs(o, None, refs)
            for k, tgt in refs:
                if tgt in idx:
                    src.append(i)
                    dst.append(idx[tgt])
                    et.append(EDGE_KEYS.index(k))
    return {"out_file": name, "x": X, "edge_index": np.array([src, dst], dtype=np.int32),
            "edge_type": np.array(et, dtype=np.int8), "node_page": node_page}

graphs, errors = [], []
with ProcessPool(max_workers=4) as pool:
    futs = {pool.schedule(to_graph, args=(n,), timeout=TIME_LIMIT): n for n in man.out_file}
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Graphs"):
        name = futs[fu]
        try:
            graphs.append(fu.result())
        except TimeoutError:
            errors.append((name, "Timeout"))
        except Exception as ex:
            errors.append((name, f"{type(ex).__name__}: {str(ex)[:70]}"))

meta_cols = ["out_file", "orig_file", "family", "variant", "label", "tool", "save_mode", "edit_type",
             "edit_page", "twin", "split"]
meta = man[meta_cols].set_index("out_file")
for g in graphs:
    g.update(meta.loc[g["out_file"]].to_dict())

with gzip.open(OUT_GRAPHS, "wb") as fh:
    pickle.dump({"graphs": graphs, "feat_names": FEAT_NAMES, "node_types": NODE_TYPES, "edge_keys": EDGE_KEYS},
                fh, protocol=pickle.HIGHEST_PROTOCOL)

n_nodes = np.array([g["x"].shape[0] for g in graphs])
n_edges = np.array([g["edge_index"].shape[1] for g in graphs])
print("\nSaved:", OUT_GRAPHS, "| size:", round(os.path.getsize(OUT_GRAPHS) / 1e6, 1), "MB")
print("Graphs:", len(graphs), "| failed:", len(errors))
if errors:
    print("Failure reasons:")
    print(pd.Series([e for _, e in errors]).value_counts().head(8).to_string())
print("Nodes per graph: median", int(np.median(n_nodes)), "| 95th pct", int(np.percentile(n_nodes, 95)), "| max", int(n_nodes.max()))
print("Edges per graph: median", int(np.median(n_edges)), "| 95th pct", int(np.percentile(n_edges, 95)), "| max", int(n_edges.max()))
print("Node features per node:", D, "| edge types:", len(EDGE_KEYS))
print("\nGraphs per version:")
print(pd.Series([g["variant"] for g in graphs]).value_counts().sort_index().to_string())

Files: 5835 | missing: 0


Graphs:   0%|          | 0/5835 [00:00<?, ?it/s]


Saved: /kaggle/working/graphs_v2.pkl.gz | size: 15.2 MB
Graphs: 5835 | failed: 0
Nodes per graph: median 86 | 95th pct 1508 | max 12644
Edges per graph: median 171 | 95th pct 3617 | max 34528
Node features per node: 43 | edge types: 12

Graphs per version:
F1_digit_resave        389
F2_digit_recompress    389
F3_digit_incr          389
F4_value_resave        389
F5_cover_replace       389
F6_digit_linearize     389
F7_stamp_fake          389
F8_textbox_fake        389
G0_original            389
G1_resave              389
G2_recompress          389
G3_note_incr           389
G4_linearize           389
G5_stamp_honest        389
G6_textbox_honest      389


In [9]:
# CELL 13 — GNN experiments on v2 graphs: GIN vs Relational-GIN (with page-level "suspicious node" readout) vs Daksh-style features vs LightGBM, plus fusion & localization
!pip install -q torch_geometric

import gzip, pickle, time, zlib, warnings
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GINEConv, global_mean_pool, global_max_pool
from torch_geometric.utils import scatter
warnings.filterwarnings("ignore")
torch.set_num_threads(4)

with gzip.open("/kaggle/working/graphs_v2.pkl.gz", "rb") as fh:
    G = pickle.load(fh)
graphs, FEAT, NE = G["graphs"], G["feat_names"], len(G["edge_keys"])
print("Graphs loaded:", len(graphs), "| node features:", len(FEAT), "| edge types:", NE)

feat = pd.read_csv("/kaggle/working/pilot_v2_features.csv")
assert len(feat) == len(graphs), "features CSV and graphs do not match"

ALL_COLS = list(range(len(FEAT)))
DAKSH_COLS = [i for i, n in enumerate(FEAT) if n.startswith("t_") or n in
              ("raw_len_log", "f_flate", "f_lzw", "f_dct", "f_none", "f_other", "entropy")]
print("Daksh-style features:", len(DAKSH_COLS), "| full features:", len(ALL_COLS))

# ---------- splits (same documents as CELL 11; 15% of training documents held out for early stopping) ----------
info = pd.DataFrame([{k: g[k] for k in ["out_file", "orig_file", "family", "variant", "label", "tool", "twin",
                                         "edit_type", "edit_page", "split"]} for g in graphs])
info["is_val"] = (info.split == "train") & info.orig_file.map(lambda s: zlib.crc32(s.encode()) % 100 < 15)
tr_idx = info.index[(info.split == "train") & ~info.is_val].tolist()
va_idx = info.index[info.is_val].tolist()
te_idx = info.index[info.split == "test"].tolist()
print(f"Train graphs: {len(tr_idx)} | validation: {len(va_idx)} | test: {len(te_idx)}")

def make_data(g, cols):
    x = torch.tensor(g["x"][:, cols].astype(np.float32))
    if x.shape[0] == 0:
        x = torch.zeros((1, len(cols)))
    ei = torch.tensor(g["edge_index"].astype(np.int64))
    et = torch.tensor(g["edge_type"].astype(np.int64))
    if ei.numel():
        ei = torch.cat([ei, ei.flip(0)], 1)
        et = torch.cat([et, et + NE])
    else:
        ei = torch.zeros((2, 0), dtype=torch.long)
        et = torch.zeros(0, dtype=torch.long)
    npg = torch.tensor(g["node_page"].astype(np.int64)) if len(g["node_page"]) else torch.full((1,), -1)
    return Data(x=x, edge_index=ei, edge_type=et, y=torch.tensor([float(g["label"])]), node_page=npg)

R = 5.0
class GNN(nn.Module):
    def __init__(self, d_in, relational, mil, hid=64, layers=3, drop=0.2):
        super().__init__()
        self.relational, self.mil, self.drop = relational, mil, drop
        self.inp = nn.Linear(d_in, hid)
        if relational:
            self.etype = nn.Embedding(2 * NE, 16)
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, hid))
            self.convs.append(GINEConv(mlp, edge_dim=16) if relational else GINConv(mlp))
            self.norms.append(nn.LayerNorm(hid))
        d_cat = hid * (layers + 1)
        if mil:
            self.node_head = nn.Sequential(nn.Linear(d_cat, hid), nn.ReLU(), nn.Linear(hid, 1))
        else:
            self.head = nn.Sequential(nn.Linear(d_cat * 2, hid), nn.ReLU(), nn.Dropout(drop), nn.Linear(hid, 1))

    def forward(self, b):
        h = F.relu(self.inp(b.x))
        hs = [h]
        ea = self.etype(b.edge_type) if self.relational else None
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, b.edge_index, ea) if self.relational else conv(h, b.edge_index)
            h = F.dropout(F.relu(norm(h)), self.drop, self.training)
            hs.append(h)
        H = torch.cat(hs, 1)
        if self.mil:
            z = self.node_head(H).squeeze(-1)
            m = scatter(z, b.batch, dim=0, reduce="max")
            s = scatter(torch.exp(R * (z - m[b.batch])), b.batch, dim=0, reduce="mean")
            return m + torch.log(s + 1e-12) / R, z
        pooled = torch.cat([global_mean_pool(H, b.batch), global_max_pool(H, b.batch)], 1)
        return self.head(pooled).squeeze(-1), None

@torch.no_grad()
def predict(model, items, want_nodes=False):
    model.eval()
    scores, nodes = [], []
    for b in DataLoader(items, batch_size=64, shuffle=False):
        out, z = model(b)
        scores += torch.sigmoid(out).tolist()
        if want_nodes and z is not None:
            ptr = b.ptr.tolist()
            nodes += [z[a:c].numpy() for a, c in zip(ptr[:-1], ptr[1:])]
    return np.array(scores), nodes

def train_gnn(name, cols, relational, mil, epochs=30, patience=6):
    torch.manual_seed(0)
    np.random.seed(0)
    data = [make_data(g, cols) for g in graphs]
    loader = DataLoader([data[i] for i in tr_idx], batch_size=32, shuffle=True)
    yva = info.label.values[va_idx]
    model = GNN(len(cols), relational, mil)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    best, best_state, bad, t0 = -1.0, None, 0, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        for b in loader:
            opt.zero_grad()
            out, _ = model(b)
            loss = F.binary_cross_entropy_with_logits(out, b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        va, _ = predict(model, [data[i] for i in va_idx])
        auc = roc_auc_score(yva, va)
        if auc > best + 1e-4:
            best, best_state, bad = auc, deepcopy(model.state_dict()), 0
        else:
            bad += 1
        print(f"  [{name}] epoch {ep:2d} | val AUC {auc:.3f} | best {best:.3f} | {time.time() - t0:.0f}s")
        if bad >= patience:
            break
    model.load_state_dict(best_state)
    te_scores, te_nodes = predict(model, [data[i] for i in te_idx], want_nodes=mil)
    return te_scores, te_nodes

# ---------- LightGBM reference (same features/params as CELL 11) ----------
META = ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"]
L1 = ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream", "obj_streams",
      "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs", "obj_order_increasing",
      "obj_gen_nonzero", "obj_max_over_count", "obj_redefined", "obj_newline_dict", "dict_open_space",
      "dict_close_space", "stream_kw_crlf", "endobj_newline_before", "reals_dot_zero", "has_ID", "tool_word_in_file", "n_pages"]
L23 = ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters", "cs_frac_zlib_repro", "cs_n_hdr",
       "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk", "cs_frac_len_indirect", "cs_ratio_std",
       "cs_ratio_max_z", "cs_n_sig", "cs_majority_share", "cs_n_singletons"]
L4 = ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"]
L5 = ["l5_n_annots", "l5_n_freetext", "l5_n_text_notes", "l5_n_img_xobj", "l5_n_fonts", "l5_n_std14_unembedded",
      "l5_frac_pages_multi_stream", "l5_max_streams_per_page"]
FULL = sorted(set(META + L1 + L23 + L4 + L5))
PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)
fi = feat.set_index("out_file").loc[info.out_file].reset_index()
lgbm = lgb.LGBMClassifier(**PARAMS).fit(fi.loc[info.split == "train", FULL], fi.loc[info.split == "train", "label"])
scores = {"LightGBM (FULL, CELL 11 model)": lgbm.predict_proba(fi.loc[te_idx, FULL])[:, 1]}

# ---------- train the GNNs ----------
print("\nTraining GNN-A: GIN, standard pooling (PDFObj2Vec-style baseline)")
scores["GNN-A: GIN (no link types)"], _ = train_gnn("GIN", ALL_COLS, relational=False, mil=False)
print("\nTraining GNN-B: Relational GIN + suspicious-node readout (ours)")
scores["GNN-B: Relational GIN + node readout"], nodes_B = train_gnn("R-GIN", ALL_COLS, relational=True, mil=True)
print("\nTraining GNN-C: same as B but only Daksh-style features (type, size, filter, entropy)")
scores["GNN-C: Relational GIN, Daksh-style features"], _ = train_gnn("R-GIN-Daksh", DAKSH_COLS, relational=True, mil=True)

te = info.loc[te_idx].reset_index(drop=True)
rank = lambda s: pd.Series(s).rank(pct=True).values
scores["FUSION: LightGBM + GNN-B (rank average)"] = (rank(scores["LightGBM (FULL, CELL 11 model)"]) +
                                                    rank(scores["GNN-B: Relational GIN + node readout"])) / 2

# ---------- evaluation (same protocol as CELL 11) ----------
NO_EV = ["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake", "G2_recompress", "G5_stamp_honest", "G6_textbox_honest"]
EV = ~te.variant.isin(NO_EV)
def tpr_at(y, s, fpr):
    thr = np.quantile(s[y == 0], 1 - fpr)
    return float((s[y == 1] > thr).mean())
def auc(y, s):
    try:
        return round(roc_auc_score(y, s), 3)
    except ValueError:
        return np.nan

PAIRS = [("G1_resave", "F1_digit_resave"), ("G2_recompress", "F2_digit_recompress"), ("G3_note_incr", "F3_digit_incr"),
         ("G1_resave", "F4_value_resave"), ("G1_resave", "F5_cover_replace"), ("G4_linearize", "F6_digit_linearize"),
         ("G5_stamp_honest", "F7_stamp_fake"), ("G6_textbox_honest", "F8_textbox_fake")]
rows, pair_rows = [], []
for name, s in scores.items():
    s = np.asarray(s)
    y = te.label.values
    rows.append({"Model": name, "AUC all": auc(y, s), "AUC evidence": auc(y[EV], s[EV]),
                 "PR-AUC evid": round(average_precision_score(y[EV], s[EV]), 3),
                 "Det@1%FA evid": round(tpr_at(y[EV], s[EV], 0.01), 3),
                 "Det@5%FA evid": round(tpr_at(y[EV], s[EV], 0.05), 3)})
    pr = {"Model": name}
    for g, f in PAIRS:
        m = te.variant.isin([g, f]).values
        pr[f.split("_")[0]] = auc(y[m], s[m])
    pair_rows.append(pr)

print("\n=== GNN vs LightGBM: MAIN TABLE (test set, unseen documents) ===")
print(pd.DataFrame(rows).to_string(index=False))
print("\n=== TWIN-PAIR AUC (each forgery vs its honest twin) ===")
print(pd.DataFrame(pair_rows).to_string(index=False))

print("\n=== PER FAMILY, evidence-bearing AUC ===")
fam_rows = []
for name, s in scores.items():
    s = np.asarray(s)
    r = {"Model": name}
    for fam in sorted(te.family.unique()):
        m = ((te.family == fam) & EV).values
        r[fam.replace("acrobat ", "").replace("adobe ", "")] = auc(te.label.values[m], s[m])
    fam_rows.append(r)
print(pd.DataFrame(fam_rows).to_string(index=False))

# ---------- localization with GNN-B: most suspicious page-text node -> page ----------
print("\n=== LOCALIZATION (GNN-B): is the most suspicious page-text node on the edited page? (always answers) ===")
te["gnnB_page"] = -1
for j, (i, z) in enumerate(zip(te_idx, nodes_B)):
    npg = graphs[i]["node_page"]
    if len(npg) == len(z) and (npg >= 0).any():
        cand = np.where(npg >= 0)[0]
        te.loc[j, "gnnB_page"] = int(npg[cand[np.argmax(z[cand])]])
for v in ["F1_digit_resave", "F4_value_resave", "F5_cover_replace", "F6_digit_linearize", "F2_digit_recompress"]:
    sub = te[(te.variant == v) & te.edit_page.notna() & (te.gnnB_page >= 0)]
    acc = round((sub.gnnB_page == sub.edit_page).mean() * 100, 1) if len(sub) else 0
    print(f"{v:20s}: correct page in {acc}% (n={len(sub)})")

out = te[["out_file", "orig_file", "family", "variant", "label", "tool", "edit_page", "gnnB_page"]].copy()
for name, s in scores.items():
    out[name] = s
out.to_csv("/kaggle/working/gnn_v2_test_scores.csv", index=False)
print("\nSaved: /kaggle/working/gnn_v2_test_scores.csv")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.2 MB/s eta 0:00:0000:01
Graphs loaded: 5835 | node features: 43 | edge types: 12
Daksh-style features: 24 | full features: 43
Train graphs: 3480 | validation: 750 | test: 1605

Training GNN-A: GIN, standard pooling (PDFObj2Vec-style baseline)
  [GIN] epoch  1 | val AUC 0.821 | best 0.821 | 15s
  [GIN] epoch  2 | val AUC 0.867 | best 0.867 | 29s
  [GIN] epoch  3 | val AUC 0.867 | best 0.867 | 43s
  [GIN] epoch  4 | val AUC 0.870 | best 0.870 | 57s
  [GIN] epoch  5 | val AUC 0.870 | best 0.870 | 71s
  [GIN] epoch  6 | val AUC 0.871 | best 0.871 | 85s
  [GIN] epoch  7 | val AUC 0.876 | best 0.876 | 99s
  [GIN] epoch  8 | val AUC 0.876 | best 0.876 | 113s
  [GIN] epoch  9 | val AUC 0.876 | best 0.876 | 126s
  [GIN] epoch 10 | val AUC 0.876 | best 0.876 | 140s
  [GIN] epoch 11 | val AUC 0.875 | best 0.876 | 154s
  [GIN] epoch 12 | val AUC 0.877 | best 0.8

In [1]:
# CELL 19 — Check what is really on disk (input + working)
import os, glob

def size_mb(p):
    if os.path.isfile(p):
        return os.path.getsize(p) / 1e6
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs) / 1e6

print("=== INPUT (PDP BPTP) ===")
for pat in ["/kaggle/input/**/clean_pool.csv", "/kaggle/input/**/inspection.csv", "/kaggle/input/**/pilot/manifest.csv"]:
    hits = glob.glob(pat, recursive=True)
    print(pat.split("/")[-1], "->", hits[0] if hits else "NOT FOUND")
src = glob.glob("/kaggle/input/**/govdocs_pdfs", recursive=True)
print("govdocs_pdfs ->", src[0] if src else "NOT FOUND", "| PDFs:", len(glob.glob(src[0] + "/*.pdf")) if src else 0)

print("\n=== WORKING (/kaggle/working) ===")
for p in sorted(glob.glob("/kaggle/working/*")):
    n = len(glob.glob(p + "/**/*", recursive=True)) if os.path.isdir(p) else 1
    print(f"{os.path.basename(p):32s} {size_mb(p):10.1f} MB | files: {n}")

=== INPUT (PDP BPTP) ===
clean_pool.csv -> /kaggle/input/notebooks/atharvagandhi/pdp-bptp/clean_pool.csv
inspection.csv -> /kaggle/input/notebooks/atharvagandhi/pdp-bptp/inspection.csv
manifest.csv -> /kaggle/input/notebooks/atharvagandhi/pdp-bptp/pilot/manifest.csv
govdocs_pdfs -> /kaggle/input/notebooks/atharvagandhi/pdp-bptp/govdocs_pdfs | PDFs: 5029

=== WORKING (/kaggle/working) ===
